In [ ]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path

print("transformers version:", __import__('transformers').__version__)
print("torch version:", torch.__version__)

import os

In [ ]:
MODEL_ID = "stanford-crfm/BioMedLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32,
    low_cpu_mem_usage=True,
    attn_implementation="eager"
)
model.eval()

n_layers = model.config.n_layer
hidden_size = model.config.n_embd
print(f"\nmodel successfully loaded")
print(f"Layers: {n_layers}")
print(f"Hidden size: {hidden_size}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
test_input = "The most common cause of myocardial infarction is"
inputs = tokenizer(test_input, return_tensors="pt")

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=10)

print("Input:", test_input)
print("Output:", tokenizer.decode(out[0], skip_special_tokens=True))
print("\nBasic generation confirmed to be working")

In [ ]:
hook_storage = {
    "hidden_states": {},
    "attention_maps": {},
    "logits_per_layer": {}
}
hooks = []

def make_hidden_state_hook(layer_idx):
    def hook(module, input, output):
        hidden = output[0].detach().squeeze(0)
        hook_storage["hidden_states"][layer_idx] = hidden
        with torch.no_grad():
            normed = model.transformer.ln_f(hidden)
            logits = model.lm_head(normed)
        hook_storage["logits_per_layer"][layer_idx] = logits.detach()
    return hook

for i, block in enumerate(model.transformer.h):
    h = block.register_forward_hook(make_hidden_state_hook(i))
    hooks.append(h)

print(f"Registered hooks on {n_layers} transformer blocks.")
print("Attention maps will be read directly from outputs.attentions")

In [ ]:
query = "What is the mechanism of action of metformin in type 2 diabetes?"
inputs = tokenizer(query, return_tensors="pt")
seq_len = inputs["input_ids"].shape[1]

for key in hook_storage:
    hook_storage[key].clear()

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

for i, attn in enumerate(outputs.attentions):
    hook_storage["attention_maps"][i] = attn.detach().squeeze(0)

print(f"hidden_states:    {len(hook_storage['hidden_states'])} layers")
print(f"attention_maps:   {len(hook_storage['attention_maps'])} layers")
print(f"logits_per_layer: {len(hook_storage['logits_per_layer'])} layers")

In [ ]:
n_layers = model.config.n_layer
hidden_size = model.config.n_embd
vocab_size = model.config.vocab_size

print("=== TENSOR SHAPE VERIFICATION ===\n")
all_good = True

for i in range(n_layers):
    hs = hook_storage["hidden_states"].get(i)
    am = hook_storage["attention_maps"].get(i)
    lp = hook_storage["logits_per_layer"].get(i)

    hs_ok = hs is not None and hs.shape == (seq_len, hidden_size)
    am_ok = am is not None and am.shape == (model.config.n_head, seq_len, seq_len)
    lp_ok = lp is not None and lp.shape == (seq_len, vocab_size)
    layer_ok = hs_ok and am_ok and lp_ok

    if not layer_ok:
        all_good = False

    if i % 8 == 0 or not layer_ok:
        print(f"Layer {i:2d} {'✓' if layer_ok else '✗'} | "
              f"hidden: {hs.shape if hs is not None else 'MISSING'} | "
              f"attn: {am.shape if am is not None else 'MISSING'} | "
              f"logits: {lp.shape if lp is not None else 'MISSING'}")

print(f"\nAll {n_layers} layers verified: {'PASS ✓' if all_good else 'FAIL ✗ — check above'}")

In [ ]:
DATA_PATH = "../data/processed/pubmedqa_filtered.json"
with open(DATA_PATH, "r") as f:
    pubmed_data = json.load(f)

sample = pubmed_data[0]

question = sample["query"]
context_str = " ".join(sample["supporting_abstracts"])[:500]
gold_answer = sample["gold_answer"]

prompt = f"Context: {context_str}\n\nQuestion: {question}\nAnswer:"

print("PubID:", sample["pubid"])
print("Question:", question)
print("Gold answer preview:", gold_answer[:150])
print("Context preview:", context_str[:150])
print("\nRunning instrumented forward pass on real clinical sample...")

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
seq_len = inputs["input_ids"].shape[1]

for key in hook_storage:
    hook_storage[key].clear()

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

for i, attn in enumerate(outputs.attentions):
    hook_storage["attention_maps"][i] = attn.detach().squeeze(0)

hs_ok = len(hook_storage["hidden_states"]) == n_layers
am_ok = len(hook_storage["attention_maps"]) == n_layers
lp_ok = len(hook_storage["logits_per_layer"]) == n_layers

print(f"\nTokens processed: {seq_len}")
print(f"hidden_states:    {len(hook_storage['hidden_states'])}/{n_layers} {'✓' if hs_ok else '✗'}")
print(f"attention_maps:   {len(hook_storage['attention_maps'])}/{n_layers} {'✓' if am_ok else '✗'}")
print(f"logits_per_layer: {len(hook_storage['logits_per_layer'])}/{n_layers} {'✓' if lp_ok else '✗'}")
print(f"\nFinal hidden state shape: {hook_storage['hidden_states'][n_layers-1].shape}")
print(f"Final logit shape:        {hook_storage['logits_per_layer'][n_layers-1].shape}")
print(f"Attention map shape:      {hook_storage['attention_maps'][n_layers-1].shape}")

if hs_ok and am_ok and lp_ok:
    print("\n00_model_load_test: ALL CHECKS PASSED")
else:
    print("\nSomething missing — check output above")